# 06 — Modelo CNN + LSTM que Sugiere la Jugada en Lenguaje Natural

Este notebook entrena el modelo final del proyecto: recibe los **4 frames** de un momento de la partida ($t-6\text{s}, t-4\text{s}, t-2\text{s}, t_{\text{actual}}$) y **escribe él mismo** la jugada sugerida, palabra por palabra.
Por ejemplo: `"Juega Valquiria detrás del puente izquierdo."` o `"esperar"`.

**No es un clasificador.** No hay cabeza de cartas ni cabeza de zonas ni un script que arme la frase después: la única salida de la red es la siguiente **palabra** del texto (*image captioning*). El vocabulario se aprende de los textos del dataset y el modelo decide qué escribir y cuándo terminar (`<eos>`).

1. **Datos (Notebook 04):** consume `dataset/train.jsonl` y `dataset/images/*.jpg` desde el CDN.
   - Se descartan las muestras con `label_visible_at_decision = true` (la etiqueta flotante con el nombre de la carta ya se ve en $t_{\text{decision}}$: fuga de la respuesta).
   - **Partición por video** (train / val / test): ninguna partida de validación o prueba aparece en entrenamiento.
2. **Arquitectura (CNN + LSTM + atención):**
   - **CNN (ResNet-18 preentrenada en ImageNet):** extrae un mapa de características de cada uno de los 4 frames ($16 \times 9$ celdas de la pantalla).
   - **LSTM temporal:** en cada celda recorre los 4 frames ($t-6\text{s} \rightarrow t_{\text{actual}}$) y resume cómo cambió esa zona (tropas que avanzan, elixir que sube, cartas que rotan).
   - **LSTM decodificador con atención:** en cada paso mira las celdas relevantes (la mano, el puente, la torre) y genera la siguiente palabra.
3. **Entrenamiento:** *teacher forcing* con entropía cruzada sobre las palabras del texto objetivo; el mejor modelo se elige por la coincidencia exacta del texto generado en validación.
4. **Evaluación sobre el texto generado (test):** coincidencia exacta, BLEU-4, carta correcta, zona correcta y `"esperar"`, contra dos líneas base (texto más frecuente y siempre `"esperar"`).
5. **Inferencia:** `sugerir_jugada(frames)` devuelve directamente la frase del modelo; galería de ejemplos de test y guardado del modelo en R2 (`models/06-cnn-lstm/`).

In [ ]:
import os, io, json, time, math, random
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import requests
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms.functional as TF
import boto3
from botocore.config import Config

# 1. Directorios de trabajo
WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('./working')
DATASET_DIR = WORK_DIR / 'dataset_vlm'
MODELS_DIR = WORK_DIR / 'models'

for d in [DATASET_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 2. Credenciales (.env local o Kaggle Secrets). El dataset se lee del CDN público;
#    R2 solo se usa al final para subir el modelo entrenado.
for env_candidate in [Path('.env'), Path('../.env')]:
    if env_candidate.exists():
        for line in env_candidate.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

def get_config(key: str, default: str = None) -> str:
    if os.getenv(key):
        return os.getenv(key)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        return default

R2_ENDPOINT = get_config('R2_ENDPOINT')
R2_ACCESS_KEY = get_config('R2_ACCESS_KEY')
R2_SECRET_KEY = get_config('R2_SECRET_KEY')
R2_BUCKET = get_config('R2_BUCKET', 'cr-videos')
CDN_BASE = get_config('CDN_URL', 'https://cdn-cr.eddux.dev').rstrip('/')

s3_client = None
if R2_ENDPOINT and R2_ACCESS_KEY and R2_SECRET_KEY:
    s3_client = boto3.client(
        's3',
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
        config=Config(signature_version='s3v4', max_pool_connections=32)
    )

# 3. Reproducibilidad y dispositivo
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

gpu_name = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'sin GPU'
print(f"✔ CDN: {CDN_BASE} | R2: {'conectado' if s3_client else 'sin credenciales (no se subirá el modelo)'}")
print(f'✔ PyTorch {torch.__version__} | Dispositivo: {DEVICE} ({gpu_name})')

In [ ]:
# =========================================================================
# CARGA DEL DATASET (NOTEBOOK 04) Y PARTICIÓN TRAIN / VAL / TEST POR VIDEO
# =========================================================================

EXCLUDE_LABEL_VISIBLE = True   # Fuera las muestras donde la etiqueta de la carta ya se ve en t_decision
VAL_FRACTION = 0.10            # Fracción de VIDEOS para validación
TEST_FRACTION = 0.10           # Fracción de VIDEOS para prueba

def load_cdn_jsonl(rel_path: str, local_file: Path) -> list[dict]:
    try:
        r = requests.get(f'{CDN_BASE}/{rel_path}', timeout=60)
        r.raise_for_status()
        local_file.write_text(r.text, encoding='utf-8')
    except Exception as e:
        if not local_file.exists():
            raise RuntimeError(f'No se pudo cargar {rel_path} desde el CDN ni local: {e}')
        print(f'⚠ CDN no disponible, usando copia local de {rel_path}: {e}')
    return [json.loads(l) for l in local_file.read_text(encoding='utf-8').splitlines() if l.strip()]

all_records = load_cdn_jsonl('dataset/train.jsonl', DATASET_DIR / 'train.jsonl')

records = [
    r for r in all_records
    if len(r.get('frames') or []) == 4 and r.get('target_text') and r.get('card_key') != 'unknown'
]
n_leak = 0
if EXCLUDE_LABEL_VISIBLE:
    n_before = len(records)
    records = [r for r in records if not (r.get('metadata') or {}).get('label_visible_at_decision')]
    n_leak = n_before - len(records)

# Partición por video: todas las muestras de un video caen en el mismo split
video_ids = sorted({r['video_id'] for r in records})
random.Random(SEED).shuffle(video_ids)
n_test_v = max(1, round(len(video_ids) * TEST_FRACTION))
n_val_v = max(1, round(len(video_ids) * VAL_FRACTION))
split_of_video = {}
for i, vid in enumerate(video_ids):
    split_of_video[vid] = 'test' if i < n_test_v else ('val' if i < n_test_v + n_val_v else 'train')

splits = {name: [r for r in records if split_of_video[r['video_id']] == name] for name in ('train', 'val', 'test')}

print(f'✔ Dataset cargado: {len(all_records)} muestras en train.jsonl | {len(records)} útiles '
      f'({n_leak} descartadas por etiqueta visible en t_decision)')
train_cards = {r['card_key'] for r in splits['train'] if r['card_key']}
for name, recs in splits.items():
    n_wait = sum(1 for r in recs if r['action_type'] == 'wait')
    cards = {r['card_key'] for r in recs if r['card_key']}
    unseen = sum(1 for r in recs if r['card_key'] and r['card_key'] not in train_cards)
    print(f'  • {name:<5}: {len(recs):>5} muestras de {len({r["video_id"] for r in recs}):>2} videos | '
          f'esperar: {100 * n_wait / max(1, len(recs)):.1f}% | cartas distintas: {len(cards)}'
          + (f' | jugadas con carta nunca vista en train: {unseen}' if name != 'train' else ''))

top_texts = Counter(r['target_text'] for r in splits['train']).most_common(5)
print('\nTextos objetivo más frecuentes en train:')
for txt, n in top_texts:
    print(f'  {n:>4} × "{txt}"')

In [ ]:
# =========================================================================
# DESCARGA DE LOS 4 FRAMES DE CADA MUESTRA DESDE EL CDN (CON CACHÉ REDIMENSIONADA)
# Los frames se exportaron a 540x960 en el NB04. Se guardan a 288x512 (misma
# proporción 9:16): las cartas de la mano siguen siendo legibles y el
# entrenamiento cabe en una GPU de Kaggle.
# =========================================================================

IMG_W, IMG_H = 288, 512
IMAGES_DIR = DATASET_DIR / f'images_{IMG_W}x{IMG_H}'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def local_frame_path(rel_path: str) -> Path:
    return IMAGES_DIR / Path(rel_path).name

def fetch_frame(rel_path: str) -> bool:
    out = local_frame_path(rel_path)
    if out.exists() and out.stat().st_size > 0:
        return True
    for attempt in range(3):
        try:
            r = requests.get(f'{CDN_BASE}/dataset/{rel_path}', timeout=30)
            r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)).convert('RGB')
            img.resize((IMG_W, IMG_H), Image.BILINEAR).save(out, quality=92)
            return True
        except Exception:
            time.sleep(1 + attempt)
    return False

all_frames = sorted({f for r in records for f in r['frames']})
t0 = time.time()
ok_by_frame = {}
with ThreadPoolExecutor(max_workers=32) as pool:
    for i, (rel, ok) in enumerate(zip(all_frames, pool.map(fetch_frame, all_frames)), 1):
        ok_by_frame[rel] = ok
        if i % 2000 == 0 or i == len(all_frames):
            print(f'  ⬇ {i}/{len(all_frames)} frames listos ({time.time() - t0:.0f}s)', flush=True)

missing = [f for f, ok in ok_by_frame.items() if not ok]
if missing:
    for name in splits:
        splits[name] = [r for r in splits[name] if all(ok_by_frame.get(f) for f in r['frames'])]
    print(f'⚠ {len(missing)} frames no se pudieron descargar; se quitan sus muestras.')
print(f"✔ Frames en caché: {IMAGES_DIR} | train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")

In [ ]:
# =========================================================================
# VOCABULARIO DE PALABRAS (EL MODELO ESCRIBE PALABRA POR PALABRA)
# - Una palabra = un token separado por espacios; el punto final es su propio token.
#   Así los nombres con puntos (p. ej. "P.E.K.K.A") se mantienen enteros.
# - El vocabulario sale solo de los textos de entrenamiento.
# =========================================================================

PAD, BOS, EOS, UNK = '<pad>', '<bos>', '<eos>', '<unk>'

def tokenize(text: str) -> list[str]:
    text = text.strip()
    if text.endswith('.'):
        return text[:-1].split() + ['.']
    return text.split()

def detokenize(tokens: list[str]) -> str:
    words = [t for t in tokens if t not in (PAD, BOS, EOS)]
    if words and words[-1] == '.':
        return ' '.join(words[:-1]) + '.'
    return ' '.join(words)

word_counts = Counter(tok for r in splits['train'] for tok in tokenize(r['target_text']))
itos = [PAD, BOS, EOS, UNK] + sorted(word_counts)
stoi = {w: i for i, w in enumerate(itos)}
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (stoi[t] for t in (PAD, BOS, EOS, UNK))

def encode(text: str) -> list[int]:
    return [BOS_ID] + [stoi.get(t, UNK_ID) for t in tokenize(text)] + [EOS_ID]

def decode(ids: list[int]) -> str:
    out = []
    for i in ids:
        if i == EOS_ID:
            break
        out.append(itos[i])
    return detokenize(out)

MAX_TEXT_LEN = max(len(encode(r['target_text'])) for r in records) + 2

example = splits['train'][0]['target_text']
print(f'✔ Vocabulario: {len(itos)} palabras (incluye 4 especiales) | Longitud máxima: {MAX_TEXT_LEN} tokens')
print(f'  Ejemplo: "{example}" → {tokenize(example)} → {encode(example)}')
print(f'  Ida y vuelta: "{decode(encode(example)[1:])}"')

In [ ]:
# =========================================================================
# DATASET Y DATALOADERS (4 FRAMES + TEXTO)
# Aumentos solo en train y IGUALES para los 4 frames de una muestra (la
# secuencia sigue siendo coherente en el tiempo):
#   - brillo / contraste / saturación (distintos videos y compresiones)
#   - desplazamiento y escala pequeños (distintos recortes de pantalla)
# Sin volteo horizontal: cambiaría izquierda/derecha y el orden de la mano.
# =========================================================================

BATCH_SIZE = 16
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 1)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def augment_clip(frames: torch.Tensor) -> torch.Tensor:
    # frames: [4, 3, H, W] en [0, 1]
    b = random.uniform(0.8, 1.2)
    c = random.uniform(0.8, 1.2)
    s = random.uniform(0.8, 1.2)
    frames = frames * b
    mean = frames.mean(dim=(1, 2, 3), keepdim=True)
    frames = (frames - mean) * c + mean
    gray = (0.299 * frames[:, 0:1] + 0.587 * frames[:, 1:2] + 0.114 * frames[:, 2:3])
    frames = (frames - gray) * s + gray
    dx = int(random.uniform(-0.03, 0.03) * IMG_W)
    dy = int(random.uniform(-0.02, 0.02) * IMG_H)
    scale = random.uniform(0.95, 1.05)
    frames = TF.affine(frames, angle=0.0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0])
    return frames.clamp(0.0, 1.0)

def load_clip(frame_paths: list) -> torch.Tensor:
    # Acepta rutas relativas del dataset, rutas absolutas o imágenes PIL; devuelve [4, 3, H, W] en [0, 1]
    arrs = []
    for f in frame_paths:
        if isinstance(f, Image.Image):
            img = f.convert('RGB')
        else:
            p = Path(f)
            img = Image.open(p if p.is_absolute() and p.exists() else local_frame_path(str(f))).convert('RGB')
        if img.size != (IMG_W, IMG_H):
            img = img.resize((IMG_W, IMG_H), Image.BILINEAR)
        arrs.append(np.asarray(img))
    return torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).float() / 255.0

def normalize_clip(frames: torch.Tensor) -> torch.Tensor:
    return (frames - IMAGENET_MEAN) / IMAGENET_STD

class CRDecisionDataset(Dataset):
    def __init__(self, recs: list[dict], train: bool):
        self.recs = recs
        self.train = train

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        frames = load_clip(r['frames'])
        if self.train:
            frames = augment_clip(frames)
        return normalize_clip(frames), torch.tensor(encode(r['target_text']), dtype=torch.long), i

def collate_batch(batch):
    frames = torch.stack([b[0] for b in batch])                       # [B, 4, 3, H, W]
    ids = nn.utils.rnn.pad_sequence([b[1] for b in batch], batch_first=True, padding_value=PAD_ID)
    idx = torch.tensor([b[2] for b in batch])
    return frames, ids, idx

datasets = {name: CRDecisionDataset(recs, train=(name == 'train')) for name, recs in splits.items()}
loaders = {
    name: DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=(name == 'train'), drop_last=(name == 'train'),
        num_workers=NUM_WORKERS, collate_fn=collate_batch, pin_memory=(DEVICE.type == 'cuda'),
        persistent_workers=NUM_WORKERS > 0
    )
    for name, ds in datasets.items()
}

f_b, ids_b, _ = next(iter(loaders['train']))
print(f'✔ Batch de entrenamiento: frames {tuple(f_b.shape)} | textos {tuple(ids_b.shape)} | workers: {NUM_WORKERS}')

In [ ]:
# =========================================================================
# ARQUITECTURA: CNN (POR FRAME) + LSTM TEMPORAL + LSTM DECODIFICADOR CON ATENCIÓN
#
#   4 frames ──► ResNet-18 ──► mapas 16x9xD ──► LSTM sobre t-6s..t-0s en cada celda
#                                                     │  (memoria espacio-temporal: 144 celdas)
#                                                     ▼
#   <bos> ──► LSTM decodificador ──► atención sobre las 144 celdas ──► siguiente palabra ──► ... <eos>
#
# La red solo emite palabras del vocabulario: la frase completa la escribe el modelo.
# =========================================================================

FEAT_DIM = 256     # canales por celda tras la CNN
EMB_DIM = 256      # embedding de palabras
HID_DIM = 512      # estado del LSTM decodificador
ATT_DIM = 256
DROPOUT = 0.3

class FrameEncoderCNN(nn.Module):
    def __init__(self, feat_dim: int):
        super().__init__()
        backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.body = nn.Sequential(*list(backbone.children())[:-2])    # sin avgpool ni fc
        self.proj = nn.Conv2d(512, feat_dim, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:              # [N, 3, H, W] -> [N, D, h, w]
        return self.proj(self.body(x))

class TemporalLSTM(nn.Module):
    # En cada celda de la pantalla, un LSTM recorre los 4 frames en orden y se queda
    # con el estado final: qué hay ahí ahora y cómo llegó (movimiento, aparición, rotación).
    def __init__(self, feat_dim: int, n_cells: int):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, feat_dim, batch_first=True)
        self.cell_pos = nn.Parameter(torch.randn(1, n_cells, feat_dim) * 0.02)   # dónde está la celda (izq/der, arriba/abajo)

    def forward(self, f: torch.Tensor) -> torch.Tensor:              # [B, T, D, h, w] -> [B, h*w, D]
        B, T, D, h, w = f.shape
        seq = f.flatten(3).permute(0, 3, 1, 2).reshape(B * h * w, T, D)
        out, _ = self.lstm(seq)
        return out[:, -1].reshape(B, h * w, D) + self.cell_pos

class AdditiveAttention(nn.Module):
    def __init__(self, mem_dim: int, hid_dim: int, att_dim: int):
        super().__init__()
        self.w_mem = nn.Linear(mem_dim, att_dim)
        self.w_hid = nn.Linear(hid_dim, att_dim)
        self.v = nn.Linear(att_dim, 1)

    def forward(self, mem: torch.Tensor, mem_proj: torch.Tensor, h: torch.Tensor):
        scores = self.v(torch.tanh(mem_proj + self.w_hid(h).unsqueeze(1))).squeeze(-1)   # [B, P]
        alpha = scores.softmax(dim=-1)
        context = (alpha.unsqueeze(-1) * mem).sum(dim=1)                                  # [B, D]
        return context, alpha

class CaptionDecoderLSTM(nn.Module):
    def __init__(self, vocab_size: int, mem_dim: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMB_DIM, padding_idx=PAD_ID)
        self.attention = AdditiveAttention(mem_dim, HID_DIM, ATT_DIM)
        self.gate = nn.Linear(HID_DIM, mem_dim)
        self.cell = nn.LSTMCell(EMB_DIM + mem_dim, HID_DIM)
        self.init_h = nn.Linear(mem_dim, HID_DIM)
        self.init_c = nn.Linear(mem_dim, HID_DIM)
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(HID_DIM + mem_dim, vocab_size)

    def init_state(self, mem: torch.Tensor):
        g = mem.mean(dim=1)
        return torch.tanh(self.init_h(g)), torch.tanh(self.init_c(g))

    def step(self, tok, h, c, mem, mem_proj):
        context, alpha = self.attention(mem, mem_proj, h)
        context = torch.sigmoid(self.gate(h)) * context
        h, c = self.cell(torch.cat([self.embed(tok), context], dim=-1), (h, c))
        logits = self.out(self.drop(torch.cat([h, context], dim=-1)))
        return logits, h, c, alpha

class CRPlayCaptioner(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.cnn = FrameEncoderCNN(FEAT_DIM)
        with torch.no_grad():
            _, _, fh, fw = self.cnn(torch.zeros(1, 3, IMG_H, IMG_W)).shape
        self.grid = (fh, fw)
        self.temporal = TemporalLSTM(FEAT_DIM, fh * fw)
        self.decoder = CaptionDecoderLSTM(vocab_size, FEAT_DIM)

    def encode(self, frames: torch.Tensor) -> torch.Tensor:          # [B, T, 3, H, W] -> [B, P, D]
        B, T = frames.shape[:2]
        f = self.cnn(frames.flatten(0, 1))
        f = f.view(B, T, *f.shape[1:])
        return self.temporal(f.float())

    def forward(self, frames: torch.Tensor, tokens_in: torch.Tensor) -> torch.Tensor:
        # Teacher forcing: en cada paso recibe la palabra correcta anterior
        mem = self.encode(frames)
        mem_proj = self.decoder.attention.w_mem(mem)
        h, c = self.decoder.init_state(mem)
        logits = []
        for t in range(tokens_in.size(1)):
            lg, h, c, _ = self.decoder.step(tokens_in[:, t], h, c, mem, mem_proj)
            logits.append(lg)
        return torch.stack(logits, dim=1)                             # [B, L, V]

    @torch.no_grad()
    def greedy(self, frames: torch.Tensor, max_len: int = None) -> list[list[int]]:
        max_len = max_len or MAX_TEXT_LEN
        mem = self.encode(frames)
        mem_proj = self.decoder.attention.w_mem(mem)
        h, c = self.decoder.init_state(mem)
        tok = torch.full((frames.size(0),), BOS_ID, dtype=torch.long, device=frames.device)
        seqs = []
        for _ in range(max_len):
            lg, h, c, _ = self.decoder.step(tok, h, c, mem, mem_proj)
            tok = lg.argmax(dim=-1)
            seqs.append(tok)
        return torch.stack(seqs, dim=1).tolist()

    @torch.no_grad()
    def beam_search(self, frames: torch.Tensor, beam_size: int = 3, max_len: int = None) -> list[int]:
        # Una sola muestra ([1, T, 3, H, W]); conserva las `beam_size` frases más probables
        max_len = max_len or MAX_TEXT_LEN
        mem = self.encode(frames)
        mem_proj = self.decoder.attention.w_mem(mem)
        h, c = self.decoder.init_state(mem)
        seqs = [[BOS_ID]]
        scores = torch.zeros(1, device=frames.device)
        finished = []
        for _ in range(max_len):
            k = len(seqs)
            tok = torch.tensor([s[-1] for s in seqs], device=frames.device)
            lg, h, c, _ = self.decoder.step(tok, h, c, mem.expand(k, -1, -1), mem_proj.expand(k, -1, -1))
            cand = (scores.unsqueeze(1) + lg.float().log_softmax(dim=-1)).view(-1)
            top_s, top_i = cand.topk(min(2 * beam_size, cand.numel()))
            new_seqs, new_scores, keep = [], [], []
            for s_val, flat in zip(top_s.tolist(), top_i.tolist()):
                parent, word = divmod(flat, lg.size(1))
                if word == EOS_ID:
                    finished.append((s_val / len(seqs[parent]), seqs[parent][1:]))
                else:
                    new_seqs.append(seqs[parent] + [word])
                    new_scores.append(s_val)
                    keep.append(parent)
                if len(new_seqs) == beam_size:
                    break
            if len(finished) >= beam_size or not new_seqs:
                break
            seqs = new_seqs
            scores = torch.tensor(new_scores, device=frames.device)
            h, c = h[keep], c[keep]
        if not finished:
            finished = [(scores[0].item() / len(seqs[0]), seqs[0][1:])]
        return max(finished, key=lambda x: x[0])[1] + [EOS_ID]

model = CRPlayCaptioner(len(itos)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'✔ Modelo CNN + LSTM listo: {n_params / 1e6:.1f} M parámetros | celdas por frame: {model.grid[0]}x{model.grid[1]}')

In [ ]:
# =========================================================================
# ENTRENAMIENTO (TEACHER FORCING) CON SELECCIÓN DEL MEJOR MODELO EN VALIDACIÓN
# - Pérdida: entropía cruzada sobre la siguiente palabra (con label smoothing).
# - La CNN preentrenada aprende con una tasa 5x menor que el resto.
# - Cada época se GENERA el texto de validación y se mide la coincidencia exacta;
#   el mejor checkpoint es el que mejor escribe la jugada, no el de menor pérdida.
# =========================================================================

EPOCHS = 30
LR_CNN = 1e-4
LR_REST = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS = 1
PATIENCE = 8                     # épocas sin mejorar en validación antes de parar
CHECKPOINT_PATH = MODELS_DIR / 'cr_cnn_lstm_best.pt'

def text_metrics(preds: list[str], recs: list[dict]) -> dict:
    # Compara el TEXTO generado con el texto objetivo (no hay clases que comparar)
    n = max(1, len(recs))
    plays = [(p, r) for p, r in zip(preds, recs) if r['action_type'] == 'play']
    waits_true = sum(1 for r in recs if r['action_type'] == 'wait')
    waits_pred = sum(1 for p in preds if p == 'esperar')
    waits_hit = sum(1 for p, r in zip(preds, recs) if r['action_type'] == 'wait' and p == 'esperar')
    return {
        'exact': sum(p == r['target_text'] for p, r in zip(preds, recs)) / n,
        'card': sum(p.startswith(f"Juega {r['card_name_es']} ") for p, r in plays) / max(1, len(plays)),
        'zone': sum(p.endswith(f" {r['placement_zone']}.") for p, r in plays) / max(1, len(plays)),
        'wait_recall': waits_hit / max(1, waits_true),
        'wait_precision': waits_hit / max(1, waits_pred)
    }

def generate_split(name: str, beam_size: int = 1) -> list[str]:
    model.eval()
    preds = [None] * len(splits[name])
    for frames, _, idx in loaders[name]:
        frames = frames.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            if beam_size == 1:
                outs = model.greedy(frames)
            else:
                outs = [model.beam_search(frames[i:i + 1], beam_size) for i in range(frames.size(0))]
        for i, seq in zip(idx.tolist(), outs):
            preds[i] = decode(seq)
    return preds

def run_epoch(train: bool) -> float:
    model.train(train)
    total, count = 0.0, 0
    for frames, ids, _ in loaders['train' if train else 'val']:
        frames = frames.to(DEVICE, non_blocking=True)
        ids = ids.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(frames, ids[:, :-1])
            loss = criterion(logits.float().reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1))
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        total += loss.item() * frames.size(0)
        count += frames.size(0)
    return total / max(1, count)

cnn_params = list(model.cnn.parameters())
cnn_ids = {id(p) for p in cnn_params}
optimizer = torch.optim.AdamW([
    {'params': cnn_params, 'lr': LR_CNN},
    {'params': [p for p in model.parameters() if id(p) not in cnn_ids], 'lr': LR_REST}
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(loaders['train'])
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: (s + 1) / warmup_steps if s < warmup_steps
    else 0.5 * (1 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps)))
)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING)

model_config = {
    'img_w': IMG_W, 'img_h': IMG_H, 'feat_dim': FEAT_DIM, 'emb_dim': EMB_DIM,
    'hid_dim': HID_DIM, 'att_dim': ATT_DIM, 'dropout': DROPOUT, 'max_text_len': MAX_TEXT_LEN
}
history = []
best_key, epochs_without_improve = None, 0

print(f"🚀 Entrenando {EPOCHS} épocas | {len(splits['train'])} muestras de train, {steps_per_epoch} pasos por época")
for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    train_loss = run_epoch(train=True)
    val_loss = run_epoch(train=False)
    val_m = text_metrics(generate_split('val'), splits['val'])
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, **{f'val_{k}': v for k, v in val_m.items()}})

    key = (val_m['exact'], -val_loss)
    improved = best_key is None or key > best_key
    if improved:
        best_key, epochs_without_improve = key, 0
        torch.save({'state_dict': model.state_dict(), 'itos': itos, 'config': model_config, 'epoch': epoch,
                    'val_metrics': val_m}, CHECKPOINT_PATH)
    else:
        epochs_without_improve += 1

    print(f"  Época {epoch:>2}/{EPOCHS} | loss train {train_loss:.3f} · val {val_loss:.3f} | "
          f"val exacta {100 * val_m['exact']:.1f}% · carta {100 * val_m['card']:.1f}% · zona {100 * val_m['zone']:.1f}% · "
          f"esperar {100 * val_m['wait_recall']:.0f}% | {time.time() - t_ep:.0f}s" + ('  ★' if improved else ''), flush=True)
    if epochs_without_improve >= PATIENCE:
        print(f'⏹ Sin mejora en {PATIENCE} épocas: se detiene el entrenamiento.')
        break

best_ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(best_ckpt['state_dict'])
print(f"✔ Mejor modelo: época {best_ckpt['epoch']} (val exacta {100 * best_ckpt['val_metrics']['exact']:.1f}%) → {CHECKPOINT_PATH}")

In [ ]:
# =========================================================================
# CURVAS DE ENTRENAMIENTO
# =========================================================================

ep = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ep, [h['train_loss'] for h in history], label='train')
axes[0].plot(ep, [h['val_loss'] for h in history], label='val')
axes[0].set_title('Pérdida (entropía cruzada por palabra)')
axes[0].set_xlabel('época')
axes[0].legend()
for k, lbl in [('val_exact', 'texto exacto'), ('val_card', 'carta correcta'), ('val_zone', 'zona correcta'), ('val_wait_recall', '"esperar" detectado')]:
    axes[1].plot(ep, [100 * h[k] for h in history], label=lbl)
axes[1].axvline(best_ckpt['epoch'], color='gray', ls='--', lw=1)
axes[1].set_title('Texto generado en validación (greedy)')
axes[1].set_xlabel('época')
axes[1].set_ylabel('%')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# EVALUACIÓN EN TEST SOBRE EL TEXTO GENERADO (BEAM SEARCH) + LÍNEAS BASE
# - Exacta: la frase generada es idéntica a la del jugador experto.
# - BLEU-4: parecido palabra a palabra (premia acertar parte de la frase).
# - Carta / zona: la frase nombra la carta / la zona que usó el experto (solo jugadas).
# =========================================================================

BEAM_SIZE = 3

def corpus_bleu4(preds: list[str], refs: list[str]) -> float:
    match, total = [0] * 4, [0] * 4
    pred_len, ref_len = 0, 0
    for p, r in zip(preds, refs):
        pt, rt = tokenize(p), tokenize(r)
        pred_len += len(pt)
        ref_len += len(rt)
        for n in range(1, 5):
            p_ngr = Counter(tuple(pt[i:i + n]) for i in range(len(pt) - n + 1))
            r_ngr = Counter(tuple(rt[i:i + n]) for i in range(len(rt) - n + 1))
            match[n - 1] += sum(min(c, r_ngr[g]) for g, c in p_ngr.items())
            total[n - 1] += max(0, len(pt) - n + 1)
    precisions = [(m + 1) / (t + 1) for m, t in zip(match, total)]   # suavizado +1
    bp = 1.0 if pred_len > ref_len else math.exp(1 - ref_len / max(1, pred_len))
    return bp * math.exp(sum(math.log(p) for p in precisions) / 4)

t0 = time.time()
test_preds = generate_split('test', beam_size=BEAM_SIZE)
test_refs = [r['target_text'] for r in splits['test']]
print(f'✔ {len(test_preds)} frases generadas en test en {time.time() - t0:.0f}s (beam={BEAM_SIZE})\n')

most_common_text = Counter(r['target_text'] for r in splits['train'] if r['action_type'] == 'play').most_common(1)[0][0]
rows = [
    ('Modelo CNN + LSTM', test_preds),
    (f'Base: siempre "{most_common_text}"', [most_common_text] * len(test_refs)),
    ('Base: siempre "esperar"', ['esperar'] * len(test_refs))
]
print(f"{'Sistema':<62} {'Exacta':>7} {'BLEU-4':>7} {'Carta':>7} {'Zona':>7} {'Esperar (recall/prec.)':>23}")
test_results = {}
for name, preds in rows:
    m = text_metrics(preds, splits['test'])
    m['bleu4'] = corpus_bleu4(preds, test_refs)
    test_results[name] = m
    print(f"{name:<62} {100 * m['exact']:>6.1f}% {100 * m['bleu4']:>6.1f} {100 * m['card']:>6.1f}% {100 * m['zone']:>6.1f}% "
          f"{100 * m['wait_recall']:>10.0f}% / {100 * m['wait_precision']:.0f}%")

distinct = Counter(test_preds)
print(f'\nFrases distintas generadas en test: {len(distinct)} (en las referencias: {len(set(test_refs))})')
print('Las más repetidas:')
for txt, n in distinct.most_common(5):
    print(f'  {n:>4} × "{txt}"')

In [ ]:
# =========================================================================
# INFERENCIA: 4 FRAMES → JUGADA SUGERIDA EN LENGUAJE NATURAL
# `sugerir_jugada` devuelve la frase tal como la escribe el modelo.
# Acepta rutas de imagen o imágenes PIL en orden [t-6s, t-4s, t-2s, t-0s].
# =========================================================================

GALLERY_N = 6

def sugerir_jugada(frames: list, beam_size: int = BEAM_SIZE) -> str:
    model.eval()
    clip = normalize_clip(load_clip(frames)).unsqueeze(0).to(DEVICE)
    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        return decode(model.beam_search(clip, beam_size))

gallery = random.Random(SEED).sample(splits['test'], min(GALLERY_N, len(splits['test'])))
fig, axes = plt.subplots(len(gallery), 4, figsize=(11, 4.6 * len(gallery)))
axes = np.atleast_2d(axes)
for row, r in enumerate(gallery):
    pred = sugerir_jugada(r['frames'])
    for col, (rel, lbl) in enumerate(zip(r['frames'], ['t-6s', 't-4s', 't-2s', 't-0s (decisión)'])):
        axes[row, col].imshow(Image.open(local_frame_path(rel)))
        axes[row, col].set_title(lbl, fontsize=9)
        axes[row, col].axis('off')
    ok = '✔' if pred == r['target_text'] else '✘'
    axes[row, 0].text(
        0, -0.12, f"{ok} Modelo: {pred}\n   Experto: {r['target_text']}   ({r['sample_id']})",
        transform=axes[row, 0].transAxes, fontsize=10, va='top', ha='left'
    )
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# GUARDADO DEL MODELO Y MÉTRICAS EN CLOUDFLARE R2 (models/06-cnn-lstm/)
# El checkpoint incluye pesos, vocabulario y configuración para recargarlo con
# CRPlayCaptioner(len(itos)) + load_state_dict.
# =========================================================================

UPLOAD_MODEL = True

metrics_payload = {
    'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'best_epoch': best_ckpt['epoch'],
    'config': model_config,
    'vocab_size': len(itos),
    'split_videos': {name: sorted({r['video_id'] for r in recs}) for name, recs in splits.items()},
    'split_samples': {name: len(recs) for name, recs in splits.items()},
    'val_metrics': best_ckpt['val_metrics'],
    'test_metrics': test_results,
    'history': history
}
metrics_file = MODELS_DIR / 'cr_cnn_lstm_metrics.json'
metrics_file.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False), encoding='utf-8')

if not UPLOAD_MODEL or s3_client is None:
    print(f'ℹ Modelo guardado solo en local: {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size / 1e6:.1f} MB)')
else:
    for local, key, ctype in [
        (CHECKPOINT_PATH, 'models/06-cnn-lstm/cr_cnn_lstm_best.pt', 'application/octet-stream'),
        (metrics_file, 'models/06-cnn-lstm/cr_cnn_lstm_metrics.json', 'application/json')
    ]:
        try:
            s3_client.upload_file(str(local), R2_BUCKET, key, ExtraArgs={'ContentType': ctype})
            print(f'☁ Subido a R2: s3://{R2_BUCKET}/{key} ({local.stat().st_size / 1e6:.1f} MB)')
        except Exception as e:
            print(f'⚠ Error subiendo {local.name} a R2: {e}')